In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
import matplotlib as mpl
from pathlib import Path
from typing import Any, Dict, List
import yaml

In [1]:
import pickle

with open("../../data/tune_zoo_cifar10_small_hyperparameter_10_random_seeds/NN_tune_trainable_6dae1_00000_0_model::dropout=0,model::init_type=uniform,model::nlin=tanh,optim::optimizer=adam,optim::wd=0.01,se_2021-09-25_11-19-22/params.pkl", "rb") as f:
    obj = pickle.load(f)

print(type(obj))
print(obj)

<class 'dict'>
{'model::type': 'CNN', 'model::channels_in': 3, 'model::o_dim': 10, 'model::nlin': 'tanh', 'model::dropout': 0, 'model::init_type': 'uniform', 'model::use_bias': False, 'optim::optimizer': 'adam', 'optim::lr': 0.001, 'optim::wd': 0.01, 'optim::momentum': 0.9, 'seed': 580069, 'training::batchsize': 10, 'training::epochs_train': 50, 'training::start_epoch': 1, 'training::output_epoch': 1, 'training::val_epochs': 1, 'training::idx_out': 500, 'training::checkpoint_dir': None, 'cuda': False, 'dataset::dump': PosixPath('/netscratch/dtaskiran/zoos/CIFAR10/small/tune_zoo_cifar10_small_hyperparameter_10_random_seeds/dataset.pt'), 'wandb': {'project': 'Corr: CIFAR Random Seed Small', 'api_key_file': '/netscratch/dtaskiran/wandb/wandb.key', 'log_config': False}}


In [14]:
df = pd.read_parquet(f"../../learning-dynamics/outputs/cache/long__8_cifar10_ViT.parquet")

In [15]:
counts = (
    df[[c for c in df.columns if c.startswith("config")]]
    .nunique()
    .sort_values(ascending=False)
)
print(counts)

config.scheduler::steps_per_epoch              7
config.model::embedding_dim                    7
config.training::batchsize                     7
config.seed                                    3
config.resources.cpu                           1
config.wandb.log_config                        1
config.resources.gpu                           1
config.wandb.api_key_file                      1
config.testloader::workers                     1
config.dataset::dump                           1
config.cuda                                    1
config.training::data_path                     1
config.training::checkpoint_freq               1
config.training::output_epoch                  1
config.training::epochs_train                  1
config.training::accumulate_gradients_steps    1
config.wandb.project                           1
config.model::type                             1
config.model::o_dim                            1
config.training::dataloader                    1
config.training::los

In [16]:
config_cols = [c for c in df.columns if c.startswith("config")]

In [17]:
for col in config_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- config.model::type ---
config.model::type
ViT    44247
Name: count, dtype: int64

--- config.model::o_dim ---
config.model::o_dim
10    44247
Name: count, dtype: int64

--- config.model::channels_in ---
config.model::channels_in
3    44247
Name: count, dtype: int64

--- config.model::dropout ---
config.model::dropout
0.1    44247
Name: count, dtype: int64

--- config.model::image_size ---
config.model::image_size
32    44247
Name: count, dtype: int64

--- config.model::patch_size ---
config.model::patch_size
4    44247
Name: count, dtype: int64

--- config.model::embedding_dim ---
config.model::embedding_dim
192    6321
24     6321
12     6321
96     6321
60     6321
48     6321
6      6321
Name: count, dtype: int64

--- config.model::depth ---
config.model::depth
12    44247
Name: count, dtype: int64

--- config.model::heads ---
config.model::heads
3    44247
Name: count, dtype: int64

--- config.optim::optimizer ---
config.optim::optimizer
adamw    44247
Name: count, dtype: int6

## Check hyper analysis
- compare importance score of different features with clusters of cluster analysis

In [18]:
config_path = "../../learning-dynamics/configs/reports/plot_configs.yaml"
config_path = Path(config_path)
with config_path.open("r", encoding="utf-8") as f:
    cfg: Dict[str, Any] = yaml.safe_load(f)


dataset_names: List[str] = cfg["datasets"]

dfs = []

for ds_name in dataset_names:
    df = pd.read_parquet(f"../../learning-dynamics/outputs/reports/tables/{ds_name}_cluster_curve_features_test_acc_testacc_k0.parquet")

    # Load the complete dataset of the model zoo containing the hyper parameters of the model
    df_hyp = pd.read_parquet(f"../../learning-dynamics/outputs/cache/long__{ds_name}.parquet")

    # Join it with the dataset of the clustering
    df = df.merge(df_hyp, on="model_name", how="left")

    dfs.append(df)

In [19]:
important_cols = [
    "final_val",
    "config.training::batchsize",
    "config.model::embedding_dim",
    "config.model::width",
    "config.model::dropout",
    "config.optim::wd",
]

mean_per_cluster_by_dataset = {}

for ds_name, df in zip(dataset_names, dfs):
    if "cluster" not in df.columns:
        print(f"Skipping {ds_name}: no 'cluster' column")
        continue

    selected_cols = [c for c in important_cols if c in df.columns]

    agg_dict = {
        "n_models": ("model_name", "nunique") if "model_name" in df.columns else ("cluster", "size")
    }

    for col in selected_cols:
        agg_dict[f"mean_{col}"] = (col, "mean")
        if col != "final_val":
            agg_dict[f"std_{col}"] = (col, "std")

    mean_per_cluster = (
        df.groupby("cluster")
          .agg(**agg_dict)
          .reset_index()
    )

    mean_per_cluster_by_dataset[ds_name] = mean_per_cluster

    print(f"\n=== {ds_name} ===")
    display(mean_per_cluster)


=== 1_cifar10_uniform_large ===


,cluster,n_models,mean_final_val,mean_config.training::batchsize,std_config.training::batchsize,mean_config.model::dropout,std_config.model::dropout,mean_config.optim::wd,std_config.optim::wd
0,0.0,567,0.616612,10.0,0.0,0.0,0.0,0.01,0.0
1,1.0,433,0.612451,10.0,0.0,0.0,0.0,0.01,0.0



=== 2_cifar10_large_hyp_random_seeds ===


,cluster,n_models,mean_final_val,mean_config.training::batchsize,std_config.training::batchsize,mean_config.model::dropout,std_config.model::dropout,mean_config.optim::wd,std_config.optim::wd
0,0.0,504,0.120786,10.0,0.0,0.326389,0.238048,0.006375,0.004414
1,1.0,776,0.536311,10.0,0.0,0.200387,0.245031,0.004932,0.004464



=== 3_cifar10_large_hyp_fixed_seeds ===


...

,cluster,n_models,mean_final_val,mean_config.training::batchsize,std_config.training::batchsize,mean_config.model::width,std_config.model::width,mean_config.model::dropout,std_config.model::dropout,mean_config.optim::wd,std_config.optim::wd
0,0.0,148,0.848454,111.027027,139.031104,71.175676,85.507699,0.0,0.0,0.0005,0.0
1,1.0,44,0.718825,739.272727,331.063252,38.772727,67.280908,0.0,0.0,0.0005,0.0



=== 7_cifar10_Resnet50_width_2-256 ===


,cluster,n_models,mean_final_val,mean_config.training::batchsize,std_config.training::batchsize,mean_config.model::width,std_config.model::width,mean_config.model::dropout,std_config.model::dropout,mean_config.optim::wd,std_config.optim::wd
0,0.0,54,0.766765,624.592593,345.53672,47.740741,75.920834,0.0,0.0,0.0005,0.0
1,1.0,139,0.872294,91.165468,113.97757,71.352518,85.693882,0.0,0.0,0.0005,0.0



=== 8_cifar10_ViT ===


,cluster,n_models,mean_final_val,mean_config.training::batchsize,std_config.training::batchsize,mean_config.model::embedding_dim,std_config.model::embedding_dim,mean_config.model::dropout,std_config.model::dropout,mean_config.optim::wd,std_config.optim::wd
0,0.0,115,0.696494,179.895652,176.836516,47.426087,50.405535,0.1,0.0,0.0005,0.0
1,1.0,32,0.198153,20.250000,16.491384,117.000000,60.896511,0.1,0.0,0.0005,0.0


In [20]:
q25 = df["final_val"].quantile(0.25)
q75 = df["final_val"].quantile(0.75)

df_25 = df.copy()
df_25["performance_group"] = df_25["final_val"].apply(
    lambda x: "worst_25%" if x <= q25 else ("best_25%" if x >= q75 else "middle_50%")
)

mean_per_group = (
    df_25.groupby("performance_group")
         .agg(
             mean_final_val=("final_val", "mean"),
             mean_batch_size=("config.training::batchsize", "mean"),
             std_batch_size=("config.training::batchsize", "std"),
             mean_embedding_dim=("config.model::embedding_dim", "mean"),
         )
)

display(mean_per_group)

,mean_final_val,mean_batch_size,std_batch_size,mean_embedding_dim
performance_group,,,,
best_25%,0.793873,115.459459,102.142435,88.540541
middle_50%,0.661390,216.438356,194.165274,29.424658
worst_25%,0.237376,34.162162,81.453198,102.000000
